# Arctic RL: Open Source Reinforcement Learning Backend
## A Unified RL Backend for Enterprise Post-Training — VeRL, SkyRL & PrimeRL

**Published:** June 29, 2026 &nbsp;|&nbsp; [Blog](https://www.snowflake.com/en/blog/engineering/arctic-rl-open-source-backend/) &nbsp;|&nbsp; [GitHub](https://github.com/Snowflake-AI-Research/Arctic-Platform)

Arctic RL is an open-source library that provides the **missing infrastructure layer** for RL post-training.
It owns GPU orchestration and system optimizations to deliver portable performance gains across RL
frameworks — enabling VeRL, SkyRL, and PrimeRL to all share the same optimized backend.

| Metric | Result |
|--------|--------|
| End-to-end training speedup | **3.5×** (Arctic-Text2SQL-R2, 32 H200 GPUs) |
| Actor-update acceleration (ZoRRo) | up to **6×** |
| BIRD dev accuracy (Text-to-SQL open-source recipe) | **59.92% → 70.35%** |
| Multi-hop QA accuracy | **69.6% → 72.3%** |
| Enterprise SQL benchmark score | **48.7** vs Gemini 3.1 Pro (47.9) & Claude 4.7 (47.3) |

## What You'll Learn

1. **The RL efficiency problem** — why current RL post-training wastes 80–95% of GPU compute on redundant prompt tokens
2. **Arctic RL architecture** — three-layer backend that makes system optimizations portable across frameworks
3. **ZoRRo** — Zero Redundancy Rollouts: prompt deduplication for training (Split Attention) and inference (Forest Cascade Attention)
4. **VeRL integration** — configure and use Arctic RL as VeRL's backend with one config flag
5. **SkyRL integration** — the already-merged SkyRL adapter (PR #1837)
6. **PrimeRL integration** — upcoming integration following the same thin-client pattern
7. **Production results** — benchmarks from Snowflake's real enterprise workloads

> **Hardware note:** Cells 3–11 (problem analysis, ZoRRo demo, visualizations) run anywhere.
> Cells 12–18 (framework configs) show the real `ArcticRLClientConfig` API — actual RL training
> requires H100/H200 GPU infrastructure.

In [ ]:
!pip install arctic-platform --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
})

try:
    from arctic_platform.rl import ArcticRLClientConfig
    ARCTIC_RL_AVAILABLE = True
    print('arctic-platform imported successfully')
except Exception as e:
    ARCTIC_RL_AVAILABLE = False
    print(f'Note: arctic-platform import incomplete ({e})')
    print('API patterns shown as reference code — install arctic-platform + ray to run them.')

print('Setup complete.')

---
## Part 1: The RL Efficiency Problem

RL post-training (PPO/GRPO) generates **N responses per prompt** to explore different policies.
This creates massive redundancy: the same prompt is processed N times.

```
Prompt A + Response 1 ──┐
Prompt A + Response 2 ──┤  ← Same prompt processed 8 times!
Prompt A + Response 3 ──┤     80–95% of tokens are duplicates
...                     │
Prompt A + Response 8 ──┘
```

With transformer attention's O(n²) complexity, recomputing shared prompts dominates the cost for long-context RL.

**Every major LLM workload has a unified backend — except RL:**
- Pre-training → DeepSpeed, Megatron-LM
- Inference → vLLM, SGLang
- RL post-training → ❌ no unified backend — every framework reimplements the system layer

**Arctic RL closes this gap.**

In [ ]:
def rl_batch_stats(prompt_len, response_len, n_responses):
    """Compute token redundancy and ZoRRo attention speedup for an RL training batch."""
    naive_tokens = (prompt_len + response_len) * n_responses
    unique_tokens = prompt_len + response_len * n_responses
    redundancy_pct = (naive_tokens - unique_tokens) / naive_tokens * 100

    # Attention O(n²) per sequence — naive
    naive_attn = n_responses * (prompt_len + response_len) ** 2
    # ZoRRo: prompt self-attention once + each response attends to full context
    zorro_attn = prompt_len**2 + n_responses * (prompt_len + response_len) * response_len
    attn_speedup = naive_attn / zorro_attn

    return dict(
        naive_tokens=naive_tokens, unique_tokens=unique_tokens,
        redundancy_pct=redundancy_pct, attn_speedup=attn_speedup
    )

scenarios = [
    ('Short (1K prompt)',   1_000,  200, 8),
    ('Medium (4K prompt)',  4_000,  500, 8),
    ('Long (10K prompt)',  10_000,  500, 8),
    ('Extreme (32K prompt)', 32_000, 1_000, 8),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
labels = [s[0] for s in scenarios]
redundancies = [rl_batch_stats(*s[1:])['redundancy_pct'] for s in scenarios]
speedups     = [rl_batch_stats(*s[1:])['attn_speedup']   for s in scenarios]
bar_colors   = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']

# Left: token redundancy
ax1 = axes[0]
bars = ax1.bar(labels, redundancies, color=bar_colors, edgecolor='black', linewidth=0.8)
ax1.set_ylim(0, 100)
ax1.set_ylabel('Redundant Tokens (%)')
ax1.set_title('Token Redundancy in RL Training\n(8 responses per prompt)')
ax1.axhline(80, color='red', linestyle='--', alpha=0.6, label='Blog: 80–95% range')
ax1.legend(fontsize=9)
for bar, v in zip(bars, redundancies):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 1, f'{v:.0f}%',
             ha='center', fontweight='bold', fontsize=11)
ax1.tick_params(axis='x', rotation=10)
ax1.grid(True, alpha=0.2, axis='y')

# Right: attention speedup from ZoRRo
ax2 = axes[1]
bars2 = ax2.bar(labels, speedups, color=bar_colors, edgecolor='black', linewidth=0.8)
ax2.set_ylabel('Attention Compute Speedup (×)')
ax2.set_title('ZoRRo Theoretical Speedup\n(reduction from O(n²) deduplication)')
ax2.axhline(1, color='gray', linestyle='--', alpha=0.4)
for bar, v in zip(bars2, speedups):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.1, f'{v:.1f}×',
             ha='center', fontweight='bold', fontsize=11)
ax2.tick_params(axis='x', rotation=10)
ax2.grid(True, alpha=0.2, axis='y')

plt.suptitle('Why RL Post-Training Needs a Unified Backend with ZoRRo',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Summary (8 responses per prompt):')
for name, p, r, n in scenarios:
    s = rl_batch_stats(p, r, n)
    print(f'  {name:<25} → {s["redundancy_pct"]:.0f}% token waste, '
          f'{s["attn_speedup"]:.1f}× ZoRRo attn speedup')

---
## Part 2: Arctic RL Architecture

Arctic RL **inverts** the traditional RL framework design:
- **Before Arctic RL:** each framework owns both the algorithm AND the GPU infrastructure
- **With Arctic RL:** the framework owns only the algorithm; Arctic RL owns the GPU infrastructure

This decoupling is what makes ZoRRo portable — optimizations live in the backend, not in each framework.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 11), dpi=150)
ax.set_xlim(0, 16)
ax.set_ylim(0, 11)
ax.axis('off')
fig.patch.set_facecolor('#ffffff')

def draw_card(x, y, w, h, bg_color, border_color, lw=1.2, radius=0.18):
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f'round,pad=0.0,rounding_size={radius}',
        facecolor=bg_color, edgecolor=border_color,
        linewidth=lw, zorder=2
    )
    ax.add_patch(box)

def draw_pill(x, y, text, bg, fg, fs=8.5):
    bbox = dict(boxstyle='round,pad=0.25,rounding_size=0.15', facecolor=bg, edgecolor='none')
    ax.text(x, y, text, ha='center', va='center', fontsize=fs, fontweight='bold', color=fg, bbox=bbox, zorder=4)

# Title Header
ax.text(8.0, 10.45, 'Arctic RL System Architecture', ha='center', va='center',
        fontsize=18, fontweight='bold', color='#0f172a')
ax.text(8.0, 10.15, 'Unified Post-Training Backend for Fast, Scalable Reinforcement Learning with ZoRRo Acceleration',
        ha='center', va='center', fontsize=11, color='#64748b')

# LAYER 1: RL Frameworks (CPU)
draw_card(0.4, 8.5, 15.2, 1.25, '#faf5ff', '#e9d5ff', lw=1.2)
ax.text(0.7, 9.45, 'LAYER 1 · RL FRAMEWORKS', fontsize=9, fontweight='bold', color='#7e22ce')
ax.text(3.8, 9.45, '(Client side · Drives training loop, rewards & advantage estimation)', fontsize=9, color='#a855f7')

fws = [
    (0.7, 'VeRL', 'PPO / GRPO', 'PR #6422 Pending Merge', '#ffffff', '#c084fc', '#581c87', '#fae8ff', '#86198f'),
    (5.7, 'SkyRL', 'GRPO Support', 'PR #1837 Merged', '#ffffff', '#c084fc', '#581c87', '#ecfdf5', '#065f46'),
    (10.7, 'PrimeRL & Others', 'Custom RL Loops', 'Integration In Progress', '#ffffff', '#ddd6fe', '#6b21a8', '#f3f4f6', '#4b5563'),
]
for x, title, sub, status, card_bg, border, title_c, badge_bg, badge_fg in fws:
    draw_card(x, 8.62, 4.6, 0.72, card_bg, border, lw=1.0)
    ax.text(x + 0.3, 8.98, title, fontsize=11, fontweight='bold', color=title_c)
    ax.text(x + 0.3, 8.76, sub, fontsize=9, color='#64748b')
    draw_pill(x + 3.4, 8.98, status, badge_bg, badge_fg, fs=7.5)

for cx in [3.0, 8.0, 13.0]:
    ax.annotate('', xy=(cx, 8.05), xytext=(cx, 8.48),
                arrowprops=dict(arrowstyle='->,head_width=0.25,head_length=0.25', color='#9333ea', lw=1.5))

# LAYER 2: CPU Client
draw_card(0.4, 7.35, 15.2, 0.7, '#f0f9ff', '#bae6fd', lw=1.2)
ax.text(0.7, 7.72, 'LAYER 2 · ARCTIC RL CPU CLIENT', fontsize=9, fontweight='bold', color='#0284c7')
methods = ['generate()', 'log_probs()', 'fwd_bwd()', 'step()', 'sync_weights()']
ax.text(4.2, 7.72, 'Thin adapter API · Eliminates direct GPU dependency in RL algorithms', fontsize=9, color='#64748b')

for i, m in enumerate(methods):
    draw_card(0.7 + i * 2.95, 7.42, 2.8, 0.25, '#ffffff', '#7dd3fc', lw=0.8, radius=0.08)
    ax.text(0.7 + i * 2.95 + 1.4, 7.545, m, ha='center', va='center', fontsize=9, fontweight='bold', color='#0369a1', family='monospace')

ax.annotate('', xy=(8.0, 6.95), xytext=(8.0, 7.33),
            arrowprops=dict(arrowstyle='->,head_width=0.25,head_length=0.25', color='#0284c7', lw=1.8))

# LAYER 3: ZoRRo System Optimizations
draw_card(0.4, 5.5, 15.2, 1.4, '#ecfdf5', '#a7f3d0', lw=1.2)
ax.text(0.7, 6.62, 'LAYER 3 · SYSTEM OPTIMIZATION LAYER (ZoRRo)', fontsize=9, fontweight='bold', color='#059669')
ax.text(5.5, 6.62, 'Transparent attention optimizations · Speeds up both training updates and rollout sampling', fontsize=9, color='#10b981')

draw_card(0.7, 5.62, 7.1, 0.88, '#ffffff', '#6ee7b7', lw=1.0)
ax.text(1.0, 6.25, 'ZoRRo Train · Split Attention', fontsize=11, fontweight='bold', color='#065f46')
ax.text(1.0, 6.02, 'Deduplicates shared prompt prefixes across rollout samples in a batch.', fontsize=8.8, color='#374151')
ax.text(1.0, 5.8, 'Eliminates redundant computation → Up to 6× faster actor updates', fontsize=8.8, fontweight='bold', color='#047857')
draw_pill(6.8, 6.25, 'TRAINING ACCELERATION', '#d1fae5', '#065f46', fs=7.5)

draw_card(8.2, 5.62, 7.1, 0.88, '#ffffff', '#6ee7b7', lw=1.0)
ax.text(8.5, 6.25, 'ZoRRo Inference · Forest Cascade Attention', fontsize=11, fontweight='bold', color='#065f46')
ax.text(8.5, 6.02, 'Hierarchical tree attention for shared generation prefixes.', fontsize=8.8, color='#374151')
ax.text(8.5, 5.8, 'Eliminates duplicate KV-cache memory loads during rollout sampling', fontsize=8.8, fontweight='bold', color='#047857')
draw_pill(14.3, 6.25, 'INFERENCE ACCELERATION', '#d1fae5', '#065f46', fs=7.5)

ax.annotate('', xy=(8.0, 5.1), xytext=(8.0, 5.48),
            arrowprops=dict(arrowstyle='->,head_width=0.25,head_length=0.25', color='#059669', lw=1.8))

# LAYER 4: Server Backend (GPU Execution)
draw_card(0.4, 3.4, 15.2, 1.65, '#fff7ed', '#fed7aa', lw=1.2)
ax.text(0.7, 4.78, 'LAYER 4 · SERVER BACKEND (GPU CLUSTER ENGINE)', fontsize=9, fontweight='bold', color='#ea580c')
ax.text(5.5, 4.78, 'Ray orchestrated distributed actors · Colocated or disaggregated deployment', fontsize=9, color='#f97316')

engines = [
    (0.7, 'DeepSpeed Engine', 'Actor & Policy Training', 'Forward, Backward & ZeRO Optimizers', 'DeepSpeed ZeRO-1/2/3', '#ea580c'),
    (5.7, 'ArcticInference Engine', 'Rollout Sampling (vLLM)', 'High-throughput continuous batching', 'Custom PagedAttention vLLM', '#ea580c'),
    (10.7, 'Log-Prob Engine', 'Reference Policy Scoring', 'Fast exact log-prob token evaluation', 'vLLM or DeepSpeed', '#ea580c'),
]
for x, title, subtitle, desc1, desc2, col in engines:
    draw_card(x, 3.52, 4.6, 1.15, '#ffffff', '#fdba74', lw=1.0)
    ax.text(x + 0.3, 4.42, title, fontsize=11, fontweight='bold', color='#7c2d12')
    ax.text(x + 0.3, 4.22, subtitle, fontsize=8.8, fontweight='bold', color=col)
    ax.text(x + 0.3, 3.98, f'• {desc1}', fontsize=8.2, color='#4b5563')
    ax.text(x + 0.3, 3.75, f'• {desc2}', fontsize=8.2, color='#4b5563')

draw_card(4.75, 4.02, 1.5, 0.45, '#ffedd5', '#fb923c', lw=0.8, radius=0.08)
ax.text(5.5, 4.28, 'NCCL / NVLink', ha='center', fontsize=7.5, fontweight='bold', color='#c2410c')
ax.text(5.5, 4.12, 'Weight Sync', ha='center', fontsize=7.5, color='#7c2d12')
ax.annotate('', xy=(5.7, 4.24), xytext=(6.2, 4.24), arrowprops=dict(arrowstyle='<->', color='#ea580c', lw=1.5))
ax.annotate('', xy=(4.75, 4.24), xytext=(5.3, 4.24), arrowprops=dict(arrowstyle='<->', color='#ea580c', lw=1.5))

ax.annotate('', xy=(8.0, 3.0), xytext=(8.0, 3.38),
            arrowprops=dict(arrowstyle='->,head_width=0.25,head_length=0.25', color='#ea580c', lw=1.8))

# LAYER 5: Infrastructure / Hardware Cluster
draw_card(0.4, 0.8, 15.2, 2.15, '#0f172a', '#1e293b', lw=1.5)
ax.text(0.7, 2.65, 'INFRASTRUCTURE · GPU COMPUTE CLUSTER & MODELS', fontsize=9.5, fontweight='bold', color='#38bdf8')
ax.text(6.0, 2.65, 'High-Performance Accelerated Compute for Enterprise Post-Training', fontsize=9, color='#94a3b8')

specs = [
    (0.7, 'Accelerator Hardware', 'NVIDIA H100 / H200 / A100 & B200\nMulti-node 8×GPU pods with NVLink + 3.2Tbps InfiniBand\nSupports colocated or disaggregated worker topologies'),
    (5.7, 'Model Architecture Compatibility', 'Any HuggingFace Causal LM (Llama 3, Qwen 2.5, DeepSeek, Arctic)\nFull Parameter ZeRO-3 or Parameter-Efficient LoRA fine-tuning\nNative support for Reasoning (R1-style) & Tool-Use policies'),
    (10.7, 'Snowflake Cortex Integration', 'Directly deployable to Snowflake Container Services (SPCS)\nLow-latency inference serving through Cortex endpoints\nFull auditability, role-based access control, and telemetry'),
]
for x, title, body in specs:
    draw_card(x, 0.95, 4.6, 1.5, '#1e293b', '#334155', lw=1.0)
    ax.text(x + 0.3, 2.2, title, fontsize=9.5, fontweight='bold', color='#f8fafc')
    lines = body.split('\n')
    for idx, line in enumerate(lines):
        ax.text(x + 0.3, 1.9 - idx * 0.28, line, fontsize=8.0, color='#94a3b8')

plt.tight_layout()
plt.show()


---
## Part 3: ZoRRo — Zero Redundancy Rollouts

ZoRRo has two components that together deliver the 3.5× end-to-end speedup:

### ZoRRo Train (Prompt Deduplication)
During actor updates, the same prompt appears N times in the batch. ZoRRo Train:
1. **Detects** sequences that share a prompt
2. **Packs** each unique prompt once into a deduplicated sequence
3. **Runs** the model once over the deduplicated batch with Split Attention
4. **Reconstructs** per-response logprobs/entropy in the original sample order

Result: mathematically identical to naive computation, up to **6× faster actor updates**.

### ZoRRo Inference (Forest Cascade Attention)
During rollout generation, many requests share a KV-cache prefix. FCA:
- Reads each shared prefix block **once per group** instead of once per request
- Splits each attention call into a grouped pass over shared blocks + per-request pass over unique blocks
- Rigorous mathematical reduction of partial results

Result: fewer redundant memory reads, faster decode for shared-prefix batches.

> **Enable both with one config flag** — `zorro_train.enable: true` and `use_fca: true`. No code changes required.

In [ ]:
from collections import defaultdict

def zorro_deduplicate(batch):
    """Pure-Python simulation of ZoRRo Train's prompt deduplication algorithm.
    Input: list of (prompt, response) pairs — prompts may repeat.
    """
    groups = defaultdict(list)
    for i, (prompt, response) in enumerate(batch):
        groups[prompt].append((i, response))

    wc = lambda s: len(s.split())

    naive_tokens = sum(wc(p) + wc(r) for p, r in batch)
    zorro_tokens = sum(
        wc(prompt) + sum(wc(r) for _, r in resps)
        for prompt, resps in groups.items()
    )

    naive_attn = sum((wc(p) + wc(r))**2 for p, r in batch)
    zorro_attn = sum(
        wc(prompt)**2 +
        sum((wc(prompt) + wc(r)) * wc(r) for _, r in resps)
        for prompt, resps in groups.items()
    )

    return {
        'n_sequences':     len(batch),
        'n_unique_prompts': len(groups),
        'naive_tokens':    naive_tokens,
        'zorro_tokens':    zorro_tokens,
        'token_savings_pct': (naive_tokens - zorro_tokens) / naive_tokens * 100,
        'attn_speedup':    naive_attn / zorro_attn,
        'groups':          {p: [i for i, _ in rs] for p, rs in groups.items()},
    }


# Simulate a real Text-to-SQL RL training batch (3 prompts × 8 responses total)
SCHEMA = 'using schema (customers, orders, products)'
batch = [
    (f'Translate to SQL {SCHEMA}: Top 10 customers by revenue',
     'SELECT c.id, SUM(o.amount) rev FROM customers c JOIN orders o ON c.id=o.cust GROUP BY 1 ORDER BY 2 DESC LIMIT 10'),
    (f'Translate to SQL {SCHEMA}: Top 10 customers by revenue',
     'SELECT customer_id, customer_name, SUM(total) FROM orders JOIN customers USING(id) GROUP BY 1,2 ORDER BY 3 DESC LIMIT 10'),
    (f'Translate to SQL {SCHEMA}: Top 10 customers by revenue',
     'WITH rev AS (SELECT cust_id, SUM(amount) t FROM orders GROUP BY 1) SELECT name, t FROM customers JOIN rev ON id=cust_id ORDER BY t DESC LIMIT 10'),
    (f'Translate to SQL {SCHEMA}: Products sold in the last 30 days',
     'SELECT DISTINCT p.name FROM products p JOIN orders o ON p.id=o.product_id WHERE o.order_date >= DATEADD(day,-30,CURRENT_DATE())'),
    (f'Translate to SQL {SCHEMA}: Products sold in the last 30 days',
     'SELECT DISTINCT product_name FROM products WHERE product_id IN (SELECT product_id FROM orders WHERE order_date > NOW()-INTERVAL 30 DAY)'),
    (f'Translate to SQL {SCHEMA}: Products sold in the last 30 days',
     'SELECT p.id, p.name, COUNT(*) sales FROM products p JOIN orders o USING(product_id) WHERE o.created_at >= CURRENT_DATE-30 GROUP BY 1,2'),
    (f'Translate to SQL {SCHEMA}: Monthly revenue trend this year',
     'SELECT DATE_TRUNC(month, order_date) AS month, SUM(amount) FROM orders WHERE YEAR(order_date)=YEAR(CURRENT_DATE()) GROUP BY 1 ORDER BY 1'),
    (f'Translate to SQL {SCHEMA}: Monthly revenue trend this year',
     'SELECT TO_CHAR(order_date,YYYY-MM) mo, SUM(total) FROM orders WHERE order_date>=DATE_TRUNC(year,CURRENT_DATE) GROUP BY 1 ORDER BY 1'),
]

stats = zorro_deduplicate(batch)

print('ZoRRo Train: Prompt Deduplication Demo')
print('=' * 60)
print(f'Batch size:            {stats["n_sequences"]} sequences')
print(f'Unique prompts:        {stats["n_unique_prompts"]}  (each processed ONCE by ZoRRo)')
print()
print(f'Token processing:')
print(f'  Naive:               {stats["naive_tokens"]:,} tokens')
print(f'  ZoRRo (deduped):     {stats["zorro_tokens"]:,} tokens')
print(f'  Savings:             {stats["token_savings_pct"]:.1f}%')
print()
print(f'Attention complexity:')
print(f'  Speedup:             {stats["attn_speedup"]:.1f}×')
print()
print('Prompt groups discovered:')
for i, (prompt, indices) in enumerate(stats['groups'].items()):
    print(f'  Group {i+1} (sequences {indices}): "{prompt[30:70]}..."')
print()
print('Outputs are mathematically identical to naive computation.')
print('ZoRRo reconstructs per-response logprobs in original sample order.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

prompt_lengths = [512, 1024, 2048, 4096, 8192, 16384, 32768]
xlabels = ['512', '1K', '2K', '4K', '8K', '16K', '32K']
response_len, n_resp = 512, 8

speedups, savings_pcts = [], []
for pl in prompt_lengths:
    naive_attn = n_resp * (pl + response_len)**2
    zorro_attn = pl**2 + n_resp * (pl + response_len) * response_len
    speedups.append(naive_attn / zorro_attn)

    naive_tok = n_resp * (pl + response_len)
    zorro_tok = pl + n_resp * response_len
    savings_pcts.append((naive_tok - zorro_tok) / naive_tok * 100)

# Left: speedup curve
ax1.plot(xlabels, speedups, 'b-o', linewidth=2.5, markersize=9)
ax1.fill_between(range(len(xlabels)), 1, speedups, alpha=0.15, color='blue')
ax1.axhline(6, color='red', linestyle='--', linewidth=2, label='Blog: up to 6× actor speedup')
ax1.set_xlabel('Prompt Length')
ax1.set_ylabel('Attention Compute Speedup (×)')
ax1.set_title('ZoRRo Train Speedup vs Prompt Length\n(8 responses × 512-token completions)')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, max(speedups) * 1.15)
for i, (xl, sp) in enumerate(zip(xlabels, speedups)):
    if sp >= 3:
        ax1.annotate(f'{sp:.1f}×', (i, sp), xytext=(0, 7),
                    textcoords='offset points', ha='center', fontsize=9,
                    fontweight='bold', color='darkblue')

# Right: token savings
bar_cols = ['#3498db' if v < 80 else '#e74c3c' for v in savings_pcts]
ax2.bar(xlabels, savings_pcts, color=bar_cols, edgecolor='black', linewidth=0.8)
ax2.axhline(80, color='red', linestyle='--', alpha=0.7, label='80% redundancy')
ax2.axhline(95, color='darkred', linestyle=':', alpha=0.5, label='95% redundancy')
ax2.set_xlabel('Prompt Length')
ax2.set_ylabel('Token Redundancy Eliminated (%)')
ax2.set_title('Token Savings from ZoRRo Deduplication\n(for long-context RL, 80–95% of tokens are duplicates)')
ax2.legend(fontsize=9)
ax2.set_ylim(0, 100)
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('ZoRRo Becomes More Powerful with Longer Contexts',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'At 32K context: {max(speedups):.1f}× attn speedup, {max(savings_pcts):.0f}% token savings')
print('Blog reports: up to 6× actor-update acceleration for long-context RL.')

---
## Part 4: Framework Integrations

All three frameworks use the same **thin CPU client adapter** pattern:

```
RL Framework (CPU)                     Arctic RL Backend (GPU)
──────────────────────────────────────────────────────────────
Training loop + reward scoring  →  client.generate()       → vLLM rollouts
Advantage estimation            →  client.log_probs()      → reference log-probs
Actor gradient update           →  client.fwd_bwd()        → DeepSpeed fwd+bwd
Optimizer step                  →  client.step()           → DeepSpeed optimizer
Weight sync                     →  client.sync_weights()   → NCCL DeepSpeed→vLLM
```

Enable ZoRRo with **two config flags** — no code changes required:
- `ds_worker_config: {zorro_train: {enable: true}}` → ZoRRo Train
- `arctic_inference_config: {use_fca: true}` → ZoRRo Inference (Forest Cascade Attention)

### 4.1 VeRL Integration

[VeRL](https://github.com/verl-project/verl) is Volcengine's RL framework for PPO/GRPO post-training.
The Arctic RL integration (PR [#6422](https://github.com/verl-project/verl/pull/6422)) lets VeRL delegate all
GPU work to Arctic RL — VeRL keeps ownership of its training loop, advantage estimation, and reward scoring.

**Available recipes:**
- [Simple single-GPU GSM8K starter](https://github.com/Snowflake-AI-Research/Arctic-Platform/tree/main/recipes/rl/verl/simple)
- [Text-to-SQL (BIRD + Snowflake)](https://github.com/Snowflake-AI-Research/Arctic-Platform/tree/main/recipes/rl/verl/txt2sql)
- [Long-context QA](https://github.com/Snowflake-AI-Research/Arctic-Platform/tree/main/recipes/rl/verl/long_context_qa)

In [ ]:
# ── VeRL + Arctic RL Configuration ──
# In your VeRL config YAML, set:  actor_rollout_ref.backend: arctic-rl
# VeRL passes the arctic_rl stanza to ArcticRLClientConfig internally.

VERL_CONFIG = '''
from arctic_platform.rl import ArcticRLClientConfig, create_arctic_rl_client

verl_config = ArcticRLClientConfig(
    backend='local',
    comm_protocol='ray',
    model_name='Qwen/Qwen3-32B',
    training_gpus=16,        # DeepSpeed ZeRO actor
    sampling_gpus=8,         # ArcticInference (vLLM) rollouts
    log_prob_gpus=8,         # Reference log-probs
    colocate=False,          # Disaggregated placement
    training_config={
        'dtype': 'bfloat16',
        'gradient_checkpointing': True,
        'optimizer': {'lr': 1e-6, 'lr_scheduler_type': 'cosine', 'warmup_ratio': 0.05},
    },
    arctic_inference_config={'use_fca': True},            # ZoRRo Inference
    ds_worker_config={'zorro_train': {'enable': True}},   # ZoRRo Train  ← ONE FLAG
)
client = create_arctic_rl_client(verl_config)
'''

VERL_LOOP = '''
# Standard VeRL GRPO training step (conceptual — VeRL handles this internally)
def grpo_step(client, prompts, reward_fn):

    # 1. Rollout: N responses per prompt (ZoRRo Inference / FCA active)
    rollouts = client.generate(prompts, sampling_params={'n': 8, 'max_tokens': 2048})

    # 2. Reward: CPU-side (e.g. SQL execution reward for Text2SQL)
    rewards = reward_fn(rollouts)

    # 3. Reference log-probs (ZoRRo Train deduplicates prompt processing)
    ref_logps = client.log_probs(prompts, completions=rollouts)

    # 4. Actor update: GRPO loss + backward (ZoRRo Train → up to 6× faster)
    loss_info = client.fwd_bwd(build_grpo_batch(rollouts, rewards, ref_logps))
    client.step()

    # 5. Push updated weights to ArcticInference
    client.sync_weights()
    return loss_info
'''

print('VeRL + Arctic RL  (PR #6422, pending merge)')
print('=' * 50)
print('Configuration:')
print(VERL_CONFIG)
print('Training Loop Pattern:')
print(VERL_LOOP)

if ARCTIC_RL_AVAILABLE:
    try:
        config = ArcticRLClientConfig(
            backend='local', comm_protocol='http',
            model_name='Qwen/Qwen3-32B',
            training_gpus=16, sampling_gpus=8, log_prob_gpus=8,
            training_config={'dtype': 'bfloat16'},
            arctic_inference_config={'use_fca': True},
            ds_worker_config={'zorro_train': {'enable': True}},
        )
        print(f'Config instantiated: model={config.model_name}, '
              f'train_gpus={config.training_gpus}, '
              f'sample_gpus={config.sampling_gpus}')
    except Exception as e:
        print(f'(Config requires full environment: {e})')

### 4.2 SkyRL Integration  ✓ Merged

[SkyRL](https://github.com/NovaSky-AI/SkyRL) is NovaSky's GRPO framework for reasoning model training.
The Arctic RL integration is **already merged** (PR [#1837](https://github.com/NovaSky-AI/SkyRL/pull/1837)).

Documentation: https://github.com/NovaSky-AI/SkyRL/tree/main/integrations/arctic_rl

You select the backend via a CLI flag — no changes to training loop code.

In [ ]:
SKYRL_CONFIG = '''
from arctic_platform.rl import ArcticRLClientConfig

# Colocated single-node setup (training + inference share the same GPUs)
skyrl_config = ArcticRLClientConfig(
    backend='local',
    comm_protocol='ray',
    model_name='Qwen/Qwen3-7B',
    training_gpus=4,
    sampling_gpus=4,
    log_prob_gpus=0,      # Share with sampling GPUs
    colocate=True,        # Fractional Ray resources — training + inference together
    training_config={'dtype': 'bfloat16', 'gradient_checkpointing': True},
    arctic_inference_config={'use_fca': True},           # ZoRRo Inference
    ds_worker_config={'zorro_train': {'enable': True}},  # ZoRRo Train
)
'''

SKYRL_CLI = '''
# Launch SkyRL with Arctic RL backend (--backend flag selects it):

python -m skyrl.train.grpo \\
    --config skyrl_gsm8k.yaml \\
    --actor_rollout_ref.backend arctic-rl \\
    --model.path Qwen/Qwen3-7B \\
    --data.train_files gsm8k_train.parquet

# SkyRL handles:   trajectory collection, advantage estimation, reward scoring
# Arctic RL owns:  GPU rollouts (vLLM+FCA), log-probs, actor gradient updates (ZoRRo)
'''

print('SkyRL + Arctic RL  (PR #1837 — MERGED)')
print('=' * 50)
print('Configuration (colocated, single-node):')
print(SKYRL_CONFIG)
print('CLI Launch:')
print(SKYRL_CLI)

print('The same 5 API calls as VeRL:')
api_table = [
    ('client.generate(prompts, sampling_params)',  'Rollout: N responses per prompt (ZoRRo Inference)'),
    ('client.log_probs(prompts, completions)',     'Reference: old/ref log-probabilities (ZoRRo Train)'),
    ('client.fwd_bwd(batch)',                      'Actor update: GRPO loss + backward (ZoRRo Train)'),
    ('client.step()',                              'Optimizer: apply gradient update'),
    ('client.sync_weights()',                      'Sync: push updated weights to inference engine'),
]
for call, desc in api_table:
    print(f'  {call:<48}  # {desc}')

### 4.3 PrimeRL Integration  (In Progress)

[PrimeRL](https://github.com/PRIME-RL/PRIME) is a high-performance RL framework targeting large-scale distributed training.

The Arctic RL integration follows the **same thin client adapter pattern** as VeRL and SkyRL.
When complete, PrimeRL will gain the full ZoRRo suite (Train + Inference) with one config flag —
the same portability guarantee applies across all three frameworks.

Track progress: https://github.com/Snowflake-AI-Research/Arctic-Platform

In [ ]:
PRIMERL_CONFIG = '''
# PrimeRL will use the same ArcticRLClientConfig (large-scale disaggregated setup)
from arctic_platform.rl import ArcticRLClientConfig, create_arctic_rl_client

primerl_config = ArcticRLClientConfig(
    backend='local',
    comm_protocol='ray',
    model_name='Qwen/Qwen3-32B',
    training_gpus=32,        # Large-scale actor (PrimeRL target)
    sampling_gpus=32,        # High-throughput rollout
    log_prob_gpus=16,        # Dedicated reference policy
    colocate=False,          # Fully disaggregated at scale
    arctic_inference_config={'use_fca': True},           # ZoRRo Inference
    ds_worker_config={'zorro_train': {'enable': True}},  # ZoRRo Train
)
client = create_arctic_rl_client(primerl_config)

# PrimeRL training loop follows the same 5-call API:
#   client.generate()  → client.log_probs()  → client.fwd_bwd()
#   → client.step()    → client.sync_weights()
'''
print('PrimeRL + Arctic RL  (Integration: In Progress)')
print('=' * 52)
print(PRIMERL_CONFIG)

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'Framework':          ['VeRL',                  'SkyRL',             'PrimeRL'],
    'Status':             ['PR #6422 (pending)',     'PR #1837 (merged)', 'In progress'],
    'ZoRRo Train':        ['Enabled',               'Enabled',           'Upcoming'],
    'ZoRRo Inference':    ['Enabled (FCA)',          'Enabled (FCA)',     'Upcoming'],
    'Placement':          ['Coloc or disaggregated', 'Colocated',         'Disaggregated'],
    'Available Recipes':  ['GSM8K, Text2SQL, LC-QA', 'GSM8K, GRPO',      'TBD'],
})

print('Arctic RL Framework Integration Summary')
print('=' * 85)
print(df.to_string(index=False))
print()
print('All frameworks use the same 5-call API: generate · log_probs · fwd_bwd · step · sync_weights')
print('All frameworks get ZoRRo by selecting the arctic-rl backend — no code changes required.')

---
## Part 5: Production Results

Arctic RL runs in production at Snowflake. The results below are from real enterprise workloads
and fully open-source recipes that the community can reproduce.

In [ ]:
# Figure 1 from the blog: 3.5× iteration-time reduction on 32 H200 GPUs
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

# Left: end-to-end training duration
systems   = ['VeRL\n(Baseline)', 'VeRL + Arctic RL\n(ZoRRo enabled)']
durations = [120, 36]   # ~5 days → ~36 hours
colors_l  = ['#e74c3c', '#27ae60']

bars1 = ax1.bar(systems, durations, color=colors_l, edgecolor='black',
                linewidth=1.5, width=0.42)
for bar, d in zip(bars1, durations):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
             f'{d}h', ha='center', fontsize=14, fontweight='bold')
ax1.annotate('', xy=(1, 36+2), xytext=(0, 120+2),
             arrowprops=dict(arrowstyle='<->', color='#2c3e50', lw=2.5))
ax1.text(0.5, 82, '3.5×\nfaster', ha='center', fontsize=16, fontweight='bold',
         color='#2c3e50', bbox=dict(boxstyle='round,pad=0.4', facecolor='#f9e79f',
                                     edgecolor='#2c3e50', linewidth=2))
ax1.set_ylabel('Training Duration (hours)')
ax1.set_title('Arctic-Text2SQL-R2 Training Time\n32 H200 GPUs (4 nodes)')
ax1.set_ylim(0, 140)
ax1.grid(True, alpha=0.2, axis='y')

# Right: compounding optimization waterfall
stages     = ['Baseline\n(VeRL only)', '+ Async\nPipelining', '+ ZoRRo\nTrain', '+ All Three\n(Full Arctic RL)']
norm_times = [1.0, 0.70, 0.37, 0.29]   # approximate from blog
stage_cols = ['#e74c3c', '#e67e22', '#3498db', '#27ae60']

bars2 = ax2.bar(stages, norm_times, color=stage_cols, edgecolor='black', linewidth=1.5)
for bar, v in zip(bars2, norm_times):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.012,
             f'{1/v:.1f}×', ha='center', fontsize=14, fontweight='bold')
ax2.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
ax2.set_ylabel('Relative Iteration Time  (1.0 = baseline)')
ax2.set_title('Compounding Effect of Arctic RL Optimizations\n(each builds on the previous)')
ax2.set_ylim(0, 1.25)
ax2.grid(True, alpha=0.2, axis='y')
legend_els = [
    mpatches.Patch(facecolor=c, label=s.replace('\n', ' '))
    for c, s in zip(stage_cols, stages)
]
ax2.legend(handles=legend_els, fontsize=9, loc='upper right')

plt.suptitle('Arctic RL Production Performance Results', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ── Text-to-SQL BIRD benchmark ──
labels_sql = ['Qwen3-32B\n(Base)', 'Arctic RL\nText2SQL Recipe\n(BIRD public benchmark)']
scores_sql = [59.92, 70.35]

bars1 = ax1.bar(labels_sql, scores_sql, color=['#bdc3c7', '#2980b9'],
                edgecolor='black', linewidth=1.5, width=0.42)
ax1.set_ylim(52, 75)
ax1.set_ylabel('BIRD Dev Accuracy (%)')
ax1.set_title('Text-to-SQL: BIRD Benchmark\n(Open-source recipe, publicly reproducible)')
for bar, v in zip(bars1, scores_sql):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{v:.2f}%', ha='center', fontsize=13, fontweight='bold')
ax1.annotate('', xy=(1, 70.35-0.8), xytext=(0, 59.92+0.8),
             arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2.5))
ax1.text(0.5, 67, f'+{70.35-59.92:.2f}pp\nimprovement', ha='center', fontsize=12,
         fontweight='bold', color='#27ae60',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#eafaf1'))

# ── Multi-hop QA ──
labels_qa = ['Base Model', 'Arctic RL\nMulti-hop QA\nRecipe']
scores_qa = [69.6, 72.3]

bars2 = ax2.bar(labels_qa, scores_qa, color=['#bdc3c7', '#8e44ad'],
                edgecolor='black', linewidth=1.5, width=0.42)
ax2.set_ylim(65, 76)
ax2.set_ylabel('Average Accuracy (%)')
ax2.set_title('Multi-hop QA: Average Accuracy\n(Across multi-hop reasoning benchmarks)')
for bar, v in zip(bars2, scores_qa):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{v:.1f}%', ha='center', fontsize=13, fontweight='bold')
ax2.annotate('', xy=(1, 72.3-0.4), xytext=(0, 69.6+0.4),
             arrowprops=dict(arrowstyle='->', color='#8e44ad', lw=2.5))
ax2.text(0.5, 71.2, f'+{72.3-69.6:.1f}pp', ha='center', fontsize=13,
         fontweight='bold', color='#8e44ad',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#f5eef8'))

plt.suptitle('Open-Source Training Recipes — Fully Reproducible Results',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Text-to-SQL recipe: Qwen3-32B base + GRPO + execution reward + BIRD public data')
print('Multi-hop QA recipe: standard GRPO training on public QA benchmarks')
print('Both are open-source — see github.com/Snowflake-AI-Research/Arctic-Platform/tree/main/recipes')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

models = [
    ('Gemini 2.5 Pro',                         52.1, 'frontier'),
    ('DeepSeek R1',                            50.9, 'frontier'),
    ('GPT-4.5',                                49.3, 'frontier'),
    ('Arctic-Text2SQL-R2\n(Snowflake, 32B)',   48.7, 'snowflake'),
    ('Gemini 3.1 Pro',                         47.9, 'frontier'),
    ('Claude 4.7',                             47.3, 'frontier'),
]

names   = [m[0] for m in models]
scores  = [m[1] for m in models]
colors_m = ['#e74c3c' if m[2] == 'snowflake' else '#95a5a6' for m in models]

bars = ax.barh(names, scores, color=colors_m, edgecolor='black', linewidth=0.8, height=0.52)
for bar, v in zip(bars, scores):
    ax.text(v + 0.07, bar.get_y() + bar.get_height()/2,
            f'{v}', va='center', fontweight='bold', fontsize=12)
ax.axvline(48.7, color='#e74c3c', linestyle='--', alpha=0.5, linewidth=1.5)
ax.set_xlabel('Enterprise SQL Benchmark Score  (higher = better)')
ax.set_title('Enterprise Text-to-SQL Benchmark\n'
             'Arctic-Text2SQL-R2 vs Frontier Models\n'
             '(Snowflake evaluated enterprise SQL benchmark, under tested conditions)')
ax.set_xlim(44, 55)
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()

# Size callout
ax.text(54.7, 3, '30–150×\nsmaller than\nfrontier models\n(32B params)',
        ha='right', va='center', fontsize=10, color='#e74c3c',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#fdf2f8', edgecolor='#e74c3c'))

legend_els = [
    mpatches.Patch(facecolor='#e74c3c', label='Arctic-Text2SQL-R2 (trained with Arctic RL)'),
    mpatches.Patch(facecolor='#95a5a6', label='Frontier models'),
]
ax.legend(handles=legend_els, loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

print('Arctic-Text2SQL-R2 key facts:')
print('  Model:    Qwen3-32B base, full-parameter RL training (no LoRA)')
print('  Training: Arctic RL + VeRL + GRPO + collision-resistant execution reward')
print('  Data:     Snowflake-first corpus (real DDL, docs, analytical scripts)')
print('  Scale:    32 H200 GPUs, ~36 hours with Arctic RL (was ~5 days without)')
print('  Result:   Outperforms models 30–150× its size on enterprise SQL')

---
## Part 6: Cortex Inference API

Snowflake Cortex exposes leading frontier models through **three industry-standard endpoints** —
all running within the Snowflake perimeter.

| API | Endpoint | Compatible SDK | Models |
|-----|----------|----------------|--------|
| **Chat Completions** | `/api/v2/cortex/v1/chat/completions` | OpenAI SDK | All |
| **Messages** | `/api/v2/cortex/anthropic/v1/messages` | Anthropic SDK | Claude only |
| **Inference Complete** | `/api/v2/cortex/inference:complete` | raw requests / SSE | All |

**Auth follows the [salmon-variant-pipeline](https://github.com/Snowflake-Labs/sfguides) pattern:**
PAT read from `~/.snowflake/config.toml` → `connections.myaccount.token` (falls back to `password`).

JWT (key-pair) is also supported — same `Authorization: Bearer` header,
different `X-Snowflake-Authorization-Token-Type` value (`KEYPAIR_JWT`).

In [ ]:
import json, requests
from snowflake.snowpark.context import get_active_session

session = get_active_session()
conn    = session.connection
host    = conn.host.replace('_', '-')
token   = conn.rest._token
BASE    = f'https://{host}'

ENDPOINTS = {
    'chat':      f'{BASE}/api/v2/cortex/v1/chat/completions',
    'messages':  f'{BASE}/api/v2/cortex/anthropic/v1/messages',
    'inference': f'{BASE}/api/v2/cortex/inference:complete',
}

def auth_headers():
    return {
        'Authorization': f'Snowflake Token="{token}"',
        'Content-Type':  'application/json',
        'Accept':        'application/json',
    }

print(f'Host: {host}')
print('Endpoints:')
for name, url in ENDPOINTS.items():
    print(f'  {name:<12} {url}')


### 6.1 Chat Completions API  (OpenAI-compatible)

Endpoint: `POST /api/v2/cortex/v1/chat/completions`

- Supports **all models** (Claude, Llama, Mistral, DeepSeek, Arctic)
- Use the **OpenAI SDK** — just swap `base_url` and inject `TOKEN_HEADER`
- Streaming via `.stream()` context manager

```python
# curl equivalent
curl -X POST "https://<host>/api/v2/cortex/v1/chat/completions" \
  -H "Authorization: Bearer $PAT" \
  -H "X-Snowflake-Authorization-Token-Type: PROGRAMMATIC_ACCESS_TOKEN" \
  -H "Content-Type: application/json" \
  -d '{"model":"claude-sonnet-4-6","messages":[{"role":"user","content":"..."}]}'
```

In [ ]:
def chat_completions(prompt, model='claude-sonnet-4-6', system=None,
                     stream=False, max_tokens=400):
    """Chat Completions via Cortex Inference REST API."""
    msgs = []
    if system:
        msgs.append({'role': 'system', 'content': system})
    msgs.append({'role': 'user', 'content': prompt})
    body = {'model': model, 'messages': msgs,
            'temperature': 0.2, 'max_tokens': max_tokens, 'stream': stream}
    r = requests.post(ENDPOINTS['chat'], headers=auth_headers(),
                      json=body, stream=stream, timeout=60)
    r.raise_for_status()
    if stream:
        parts = []
        for raw in r.iter_lines():
            line = raw.decode('utf-8') if isinstance(raw, bytes) else raw
            if not line.startswith('data:'): continue
            payload = line[5:].strip()
            if not payload or payload == '[DONE]': continue
            try:
                for ch in json.loads(payload).get('choices', []):
                    t = ch.get('delta', {}).get('content', '')
                    if t:
                        print(t, end='', flush=True)
                        parts.append(t)
            except Exception: continue
        print()
        return ''.join(parts)
    return r.json()['choices'][0]['message']['content']


PROMPT_CHAT = (
    'In 3 concise sentences explain what ZoRRo (Zero Redundancy Rollouts) does '
    'in Arctic RL post-training and why it matters for long-context RL workloads.'
)

print('Chat Completions — Cortex Inference REST API  (claude-sonnet-4-6, streaming)')
print('─' * 62)
result_chat = chat_completions(PROMPT_CHAT, stream=True)


### 6.2 Messages API  (Anthropic-compatible)

Endpoint: `POST /api/v2/cortex/anthropic/v1/messages`

- **Claude models only**
- Requires extra header: `anthropic-version: 2023-06-01`
- Streaming returns **SSE** — parse `content_block_delta` events from `iter_lines()`
- Falls back to chat/completions automatically on 404

```python
# curl equivalent
curl -X POST "https://<host>/api/v2/cortex/anthropic/v1/messages" \
  -H "Authorization: Bearer $PAT" \
  -H "X-Snowflake-Authorization-Token-Type: PROGRAMMATIC_ACCESS_TOKEN" \
  -H "Content-Type: application/json" \
  -H "anthropic-version: 2023-06-01" \
  -d '{"model":"claude-sonnet-4-6","max_tokens":400,"messages":[{"role":"user","content":"..."}]}'
```

In [ ]:
def messages_api(prompt, model='claude-sonnet-4-6', system=None,
                 stream=False, max_tokens=400):
    """Messages API (Anthropic-compatible) via Cortex Inference REST API."""
    hdrs = {**auth_headers(), 'anthropic-version': '2023-06-01'}
    body = {'model': model, 'max_tokens': max_tokens,
            'messages': [{'role': 'user', 'content': prompt}], 'stream': stream}
    if system:
        body['system'] = system
    r = requests.post(ENDPOINTS['messages'], headers=hdrs,
                      json=body, stream=stream, timeout=60)
    if r.status_code == 404:
        print('[WARN] 404 on messages endpoint — falling back to chat/completions')
        return chat_completions(prompt, model=model, stream=stream,
                                system=system, max_tokens=max_tokens)
    r.raise_for_status()
    if stream:
        parts = []
        for raw in r.iter_lines():
            line = raw.decode('utf-8') if isinstance(raw, bytes) else raw
            if line and line.startswith('data: '):
                chunk = json.loads(line[6:])
                if chunk.get('type') == 'content_block_delta':
                    t = chunk['delta'].get('text', '')
                    print(t, end='', flush=True)
                    parts.append(t)
        print()
        return ''.join(parts)
    return r.json()['content'][0]['text']


PROMPT_MSG = (
    'VeRL, SkyRL, and PrimeRL all integrate Arctic RL through the same thin client adapter. '
    'What are the five API calls this adapter makes, and which Arctic RL optimizations '
    'are transparently active during each call?'
)

print('Messages API — Cortex Inference REST API  (claude-sonnet-4-6, streaming)')
print('─' * 62)
result_msg = messages_api(PROMPT_MSG, stream=True)


### 6.3 Inference Complete  (Legacy / Tools)

Endpoint: `POST /api/v2/cortex/inference:complete`

- Supports all models; always returns **SSE** regardless of `stream` flag
- Parse `choices[].delta.text` events from the response body
- Use for tool calling, custom routing, or legacy integrations

```python
# curl equivalent
curl -X POST "https://<host>/api/v2/cortex/inference:complete" \
  -H "Authorization: Bearer $PAT" \
  -H "X-Snowflake-Authorization-Token-Type: PROGRAMMATIC_ACCESS_TOKEN" \
  -H "Content-Type: application/json" \
  -d '{"model":"claude-sonnet-4-6","messages":[{"role":"user","content":"..."}],"max_tokens":400}'
```

In [ ]:
def inference_complete(prompt, model='claude-sonnet-4-6', system=None, max_tokens=400):
    """inference:complete endpoint — always returns SSE."""
    msgs = []
    if system:
        msgs.append({'role': 'system', 'content': system})
    msgs.append({'role': 'user', 'content': prompt})
    body = {'model': model, 'messages': msgs,
            'temperature': 0.2, 'max_tokens': max_tokens}
    r = requests.post(ENDPOINTS['inference'], headers=auth_headers(),
                      json=body, timeout=60)
    r.raise_for_status()
    parts = []
    for line in r.text.splitlines():
        if not line.startswith('data:'): continue
        payload = line[5:].strip()
        if not payload or payload == '[DONE]': continue
        try:
            for ch in json.loads(payload).get('choices', []):
                d = ch.get('delta', {})
                if d.get('type') == 'text':
                    parts.append(d.get('text', ''))
        except Exception: continue
    return ''.join(parts)


PROMPT_INF = 'Name the three GPU engine types in Arctic RL and describe each role in one sentence.'

print('Inference Complete — Cortex Inference REST API  (claude-sonnet-4-6)')
print('─' * 62)
result_inf = inference_complete(PROMPT_INF)
print(result_inf)


In [ ]:
import pandas as pd

summary_api = pd.DataFrame({
    'API':             ['Chat Completions',
                        'Messages',
                        'Inference Complete'],
    'Endpoint':        ['/api/v2/cortex/v1/chat/completions',
                        '/api/v2/cortex/anthropic/v1/messages',
                        '/api/v2/cortex/inference:complete'],
    'Models':          ['All  (Claude, Llama, Mistral, DeepSeek, Arctic)',
                        'Claude only',
                        'All'],
    'SDK':             ['raw requests', 'raw requests', 'raw requests'],
    'Streaming':       ['SSE iter_lines() → choices[].delta.content', 'SSE iter_lines() → content_block_delta', 'SSE always → choices[].delta.text'],
    'Extra header':    ['—', 'anthropic-version: 2023-06-01', '—'],
})

print('Cortex REST API — Endpoint Reference')
print('=' * 72)
print(summary_api.to_string(index=False))
print()
print('Auth headers (all three endpoints):')
print('  Authorization: Bearer <token>')
print('  X-Snowflake-Authorization-Token-Type: PROGRAMMATIC_ACCESS_TOKEN  # PAT')
print('                                        KEYPAIR_JWT                 # JWT / key-pair')
print('                                        OAUTH                       # OAuth token')
print()
print('PAT (recommended — simplest setup):')
print('  cfg = tomllib.loads((Path.home() / ".snowflake/config.toml").read_text())')
print('  PAT = cfg["connections"]["myaccount"].get("token") or cfg["connections"]["myaccount"].get("password", "")')
print()
print('JWT (key-pair — production / service accounts):')
print('  iss = "ACCOUNT.USER.SHA256:<fingerprint>"')
print('  sub = "ACCOUNT.USER"')
print('  exp = now + 3600  (max 1 hour)')
print('  jwt.encode({iss, sub, iat, exp}, rsa_key, algorithm="RS256")')

---
## Summary

### What Arctic RL provides
| Layer | What it does |
|-------|--------------|
| **Server Backend** | DeepSpeed training + ArcticInference (vLLM) sampling + log-prob engine, Ray-orchestrated |
| **ZoRRo Train** | Prompt deduplication via Split Attention → up to **6× actor-update acceleration** |
| **ZoRRo Inference** | Forest Cascade Attention in decode → fewer redundant KV reads during rollouts |
| **CPU Client** | Thin adapter (5 calls) for any RL framework — no GPU awareness needed |

### Framework status
| Framework | Status | Enable ZoRRo |
|-----------|--------|--------------|
| **VeRL** | PR #6422, pending merge | `ds_worker_config: {zorro_train: {enable: true}}` |
| **SkyRL** | PR #1837, **merged** | same config flag |
| **PrimeRL** | In progress | same config flag |

### Getting started
```bash
pip install arctic-platform

# Or from source:
git clone https://github.com/Snowflake-AI-Research/Arctic-Platform.git
pip install -e '.[rl]'
```

See the [VeRL recipes](https://github.com/Snowflake-AI-Research/Arctic-Platform/tree/main/recipes/rl/verl)
for runnable Text-to-SQL and long-context QA examples.